In [1]:
import numpy as np
import pandas as pd

from exerpy import ExergyAnalysis
from exerpy.analyses import _load_json

In [2]:
def diff_between_simulators(testcase):
    simulator_results = []
    for path in testcase.values():
        contents = _load_json(path)
        if "settings" not in contents:
            contents["settings"] = {}

        simulator_results += [
            ExergyAnalysis.from_json(path, **contents["settings"])
        ]

    sim1 = simulator_results[0]
    sim2 = simulator_results[1]

    columns = ["m", "p", "T"]
    if sim1.chemExLib is not None:
        columns.append("e_CH")
    if sim1.split_physical_exergy:
        columns.append("e_M")
        columns.append("e_T")
    else:
        columns.append("e_PH")

    df_sim1 = pd.DataFrame.from_dict(
        sim1._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")
    df_sim2 = pd.DataFrame.from_dict(
        sim2._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")

    overlapping_index = list(
        set(df_sim1.index.tolist()) & set(df_sim2.index.tolist())
    )
    df_sim1 = df_sim1.loc[overlapping_index].round(6)
    df_sim2 = df_sim2.loc[overlapping_index].round(6)

    # inf means that sim2 has 0 value, comparison does not make sense there
    # and sometimes there seem to be NaN values in the dataframes, those are
    # removed as well
    diff = ((df_sim1 - df_sim2) / df_sim2).abs().replace(np.inf, 0).fillna(0)
    # exergies near the ambient state are close to zero, so differences below 1 J/kg are round-off
    exergies = [column for column in columns if column.startswith("e_")]
    diff[exergies] = diff[exergies].mask((df_sim1 - df_sim2)[exergies].abs() < 1.0, 0)
    return diff

In [3]:
testcase = {
    "ebsilon": "hp_ebs.json",
    "tespy": "hp_tespy.json"
}
diff_between_simulators(testcase)

,m,p,T,e_M,e_T
13,0.000002,0.000000e+00,0.000000e+00,0.0,0.000000
12,0.000002,9.971237e-10,1.134148e-05,0.0,0.000000
33,0.000079,1.454546e-10,1.154332e-05,0.0,0.000000
11,0.000002,0.000000e+00,0.000000e+00,0.0,0.000000
34,0.000079,0.000000e+00,3.992982e-05,0.0,0.000063
23,0.000000,0.000000e+00,0.000000e+00,0.0,0.000000
32,0.000079,0.000000e+00,3.625946e-09,0.0,0.000000
21,0.000000,0.000000e+00,0.000000e+00,0.0,0.000000
31,0.000079,0.000000e+00,0.000000e+00,0.0,0.000148
22,0.000000,0.000000e+00,0.000000e+00,0.0,0.000000


In [4]:
testcase = {
    "tespy": "hp_tespy.json",
    "aspen": "hp_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_M,e_T
13,0.000483,0.0,0.000000e+00,0.0,0.000000
12,0.000483,0.0,9.652330e-07,0.0,0.000000
33,0.000093,0.0,1.231574e-06,0.0,0.000000
11,0.000483,0.0,0.000000e+00,0.0,0.000000
34,0.000093,0.0,4.539902e-05,0.0,0.000075
23,0.000000,0.0,2.543558e-09,0.0,0.000000
32,0.000093,0.0,2.538162e-08,0.0,0.000000
21,0.000000,0.0,0.000000e+00,0.0,0.000000
31,0.000093,0.0,2.872321e-09,0.0,0.000156
22,0.000000,0.0,2.914173e-09,0.0,0.000000


In [5]:
testcase = {
    "ebsilon": "hp_ebs.json",
    "aspen": "hp_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_M,e_T
13,0.000485,0.000000e+00,0.000000e+00,0.0,0.0
12,0.000485,9.971237e-10,1.037626e-05,0.0,0.0
33,0.000013,1.454546e-10,1.031176e-05,0.0,0.0
11,0.000485,0.000000e+00,0.000000e+00,0.0,0.0
34,0.000013,0.000000e+00,5.471011e-06,0.0,0.0
23,0.000000,0.000000e+00,2.543558e-09,0.0,0.0
32,0.000013,0.000000e+00,2.900757e-08,0.0,0.0
21,0.000000,0.000000e+00,0.000000e+00,0.0,0.0
31,0.000013,0.000000e+00,2.872321e-09,0.0,0.0
22,0.000000,0.000000e+00,2.914173e-09,0.0,0.0


In [6]:
# Component results stored by the example scripts hp_ebs.py, hp_tespy.py and hp_aspen.py
results = {
    simulator: pd.read_csv(f"hp_components_{simulator.lower()}.csv", index_col=0).set_index("Component")
    for simulator in ("Ebsilon", "TESPy", "Aspen")
}

# Components the simulators model differently are compared as one group
groups = {
    "PUMP-MOTOR": "MOT3",  # Aspen name of the pump motor
}

# Exergy destruction in kW of the components found in at least two models, and the deviations from Ebsilon
E_D = (
    pd.DataFrame({simulator: df["E_D [kW]"] for simulator, df in results.items()})
    .drop("TOT")
    .rename(index=groups)
    .groupby(level=0)
    .sum(min_count=1)
    .dropna(thresh=2)
)
reference = E_D["Ebsilon"].where(E_D["Ebsilon"].abs() > 1e-3)  # no relative deviation below 1 W
for simulator in ("TESPy", "Aspen"):
    E_D[f"{simulator} vs Ebsilon [%]"] = (E_D[simulator] / reference - 1) * 100
E_D.sort_values("Ebsilon", ascending=False).round(2)

,Ebsilon,TESPy,Aspen,TESPy vs Ebsilon [%],Aspen vs Ebsilon [%]
Component,,,,,
EVA,647.20,647.06,647.47,-0.02,0.04
VAL,144.57,144.61,144.57,0.02,-0.00
COMP,125.81,125.82,125.81,0.00,-0.00
FAN,104.88,104.86,104.93,-0.02,0.05
COND,91.49,91.48,91.50,-0.01,0.00
MOT2,13.61,13.62,13.61,0.00,-0.00
MOT1,10.74,10.74,10.75,-0.02,0.05
PUMP,0.01,0.01,0.01,-0.21,-0.21
MOT3,0.00,0.00,0.00,NaN,NaN


In [7]:
# Results of the whole plant
columns = ["E_F [kW]", "E_P [kW]", "E_D [kW]", "E_L [kW]", "epsilon [%]"]
pd.DataFrame({simulator: df.loc["TOT", columns] for simulator, df in results.items()}).round(3)

,Ebsilon,TESPy,Aspen
E_F [kW],1624.046,1623.907,1624.384
E_P [kW],483.918,483.918,483.918
E_D [kW],1138.332,1138.192,1138.658
E_L [kW],1.796,1.796,1.808
epsilon [%],29.797,29.800,29.791
